In [ ]:
# data
import os

db_url = os.getenv("DB_URL")

import pandas as pd
from sqlalchemy import create_engine

db_url = db_url.replace("postgres://", "postgresql+psycopg://")
engine = create_engine(db_url)

query = "SELECT * FROM RAW_DATA;"
df = pd.read_sql(query, engine)

df = df.pivot(index='mz', columns='sample_id', values='intensity')

import matplotlib.pyplot as plt
from scipy.signal import find_peaks

mean_intensity = df.mean(axis=1)
std_intensity = df.std(axis=1)
upper_bound = mean_intensity + (3 * std_intensity)
lower_bound = mean_intensity - (3 * std_intensity)

current_run = df.iloc[:, -1]
out_of_bounds = (current_run > upper_bound) | (current_run < lower_bound)

breach_peaks, _ = find_peaks(current_run.to_numpy(), height=upper_bound.to_numpy())

plt.figure(figsize=(12, 6))
plt.fill_between(df.index, lower_bound, upper_bound, color='gold', alpha=0.3, label='Golden Corridor')
plt.plot(df.index, current_run, color='black', label='Current Run')

plt.scatter(df.index[breach_peaks], current_run.iloc[breach_peaks], color='red', label='Out of Spec')

plt.legend()
plt.show()

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from utils.db import sql_select
import pandas as pd 

query = "SELECT \
    (pg_size_pretty(pg_database_size(current_database()))) AS memory, \
    (SELECT COUNT(id) FROM samples) AS s, \
    (SELECT COUNT(DISTINCT sample_id) FROM ms_data) AS ms, \
    (SELECT COUNT(DISTINCT sample_id) FROM fab_data) AS fab, \
    (SELECT COUNT(DISTINCT sample_id) FROM raw_data) AS rw, \
    (SELECT COUNT(*) FROM raw_data) AS full_rw;"
status = sql_select(query)
status

quering SELECT     (pg_size_pretty(pg_database_size(current_database()))) AS memory,     (SELECT COUNT(id) FROM samples) AS s,     (SELECT COUNT(DISTINCT sample_id) FROM ms_data) AS ms,     (SELECT COUNT(DISTINCT sample_id) FROM fab_data) AS fab,     (SELECT COUNT(DISTINCT sample_id) FROM raw_data) AS rw,     (SELECT COUNT(*) FROM raw_data) AS full_rw;


,memory,s,ms,fab,rw,full_rw
0,57 MB,248,248,248,245,367500


In [3]:
# data
import sys
import os

sys.path.append(os.path.abspath('..'))

import pickle
import pandas as pd
from autogluon.tabular import TabularPredictor as tr
from utils.db import sql_execute, sql_select, db_url

query = \
    "SELECT \
    s.*, \
    f.timestamp, f.temperature, f.pressure, f.ph_level, \
    m.mass_to_charge, m.charge, m.intensity \
    FROM samples s \
    JOIN fab_data f ON f.sample_id = s.id \
    JOIN ms_data m ON m.sample_id = s.id \
    ORDER BY s.analysis_date;"
raw_data = sql_select("SELECT * FROM RAW_DATA;") 
processed_data = sql_select(query)

#---------------------------------------------------------------------------------

peak_sums = processed_data.groupby('id')['intensity'].sum().reset_index()

peak_sums['is_good'] = (peak_sums['intensity'] > 17000).astype(int)

fab_features = processed_data[['id', 'temperature', 'pressure', 'ph_level']].drop_duplicates()

ml_df = pd.merge(fab_features, peak_sums[['id', 'is_good']], on='id').drop(columns=['id'])

predictor = tr(label='is_good', eval_metric='f1').fit(ml_df, time_limit=120)

predictor.leaderboard()

ModuleNotFoundError: No module named 'autogluon'

In [ ]:
import lightgbm as lgb
from sklearn.neural_network import MLPClassifier as mlpc
from sklearn.preprocessing import StandardScaler as ss
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import mlflow
from utils.db import mlflow_db_url

mlflow.set_tracking_uri(mlflow_db_url)
mlflow.set_experiment("Paracetamol_MS_Quality")

X = ml_df[['temperature', 'pressure', 'ph_level']]
y = ml_df['is_good']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

mlflow.lightgbm.autolog()
with mlflow.start_run(run_name="LightGBM_MS_v1"):
    lgb_model = lgb.LGBMClassifier(random_state=42)
    lgb_model.fit(X_train, y_train)
    lgb_preds = lgb_model.predict(X_test)
    mlflow.log_metric("f1_score_test", f1_score(y_test, lgb_preds))

mlflow.sklearn.autolog()
with mlflow.start_run(run_name="MLPC_MS_v1"):
    scaler = ss()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    nn_model = mlpc(random_state=42, max_iter=500)
    nn_model.fit(X_train_scaled, y_train)

    nn_preds = nn_model.predict(X_test_scaled)
    mlflow.log_metric("f1_score_test", f1_score(y_test, nn_preds))

def save_trained_model(name, model):
    binary_model = pickle.dumps(model)

    insert_query = "INSERT INTO models (name, model_binary) VALUES (%s, %s)"
   
    print(f"Inserting model: {name}")
    sql_execute(insert_query, (name, binary_model,))
    

save_trained_model("MLPC MS v1.1", nn_model)
save_trained_model("Lightgbm MS v1.1", lgb_model)
models = sql_select("SELECT * FROM models;")
models